# Few-Shot Day-to-Night Translation: Qualitative Comparison

This notebook creates reproducible qualitative grids for the three pre-defined sample roles. Every comparison uses the same filename, training seed 1, inference seed 0, and the 5-, 10-, and 50-shot checkpoints. The figures are descriptive examples rather than a formal human evaluation.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from PIL import Image


def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'results' / 'qualitative_selection' / 'selection.csv').is_file():
            return candidate
    raise FileNotFoundError('Could not locate results/qualitative_selection/selection.csv')


ROOT = find_project_root()
SELECTION_DIR = ROOT / 'results' / 'qualitative_selection'
GENERATED_ROOT = ROOT / 'results' / 'qualitative_generated'
FIGURE_DIR = ROOT / 'results' / 'qualitative_figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

test_candidates = [
    ROOT / 'data' / 'processed' / 'test_main200',
    ROOT / 'data' / 'processed' / 'test',
]
TEST_ROOT = next((path for path in test_candidates if path.is_dir()), None)
if TEST_ROOT is None:
    raise FileNotFoundError('Neither test_main200 nor the full processed test root exists')

selection = pd.read_csv(SELECTION_DIR / 'selection.csv').set_index('role')
ROLE_ORDER = ['typical', 'challenging', 'breaking_sensitive']
ROLE_LABELS = {
    'typical': 'Typical',
    'challenging': 'Challenging',
    'breaking_sensitive': 'Breaking-sensitive',
}
assert set(ROLE_ORDER).issubset(selection.index)

display_columns = [
    'filename', 'difficulty_score', 'mean_ssim', 'mean_lpips',
    'mean_clip_similarity', 'pix2pix_5_to_10_lpips_degradation',
    'cyclegan_10_to_20_clip_degradation',
]
display(selection.loc[ROLE_ORDER, display_columns].round(4))
print(f'Input/target root: {TEST_ROOT}')
print(f'Figure output: {FIGURE_DIR}')

## Deterministic selection rules

- **Typical:** closest to the median standardized difficulty, where $D=z(\mathrm{LPIPS})-z(\mathrm{SSIM})-z(\mathrm{CLIP})$.
- **Challenging:** closest to the 90th percentile of $D$; this avoids selecting a single extreme outlier.
- **Breaking-sensitive:** largest combined Pix2Pix 5-to-10 LPIPS degradation and CycleGAN 10-to-20 CLIP degradation, preferring changes consistent in at least two of three seeds.

CMMD is not used for image selection because it is computed at the generated-set level rather than per image.

In [ ]:
PIX_COLOR = '#2C7FB8'
CYCLE_COLOR = '#E76F51'
REFERENCE_COLOR = '#6B7280'

CONDITIONS = [
    ('Input\n(day)', 'input', None, REFERENCE_COLOR),
    ('Target\n(night)', 'target', None, REFERENCE_COLOR),
    ('Pix2Pix\n5-shot', 'generated', ('pix2pix', 5), PIX_COLOR),
    ('Pix2Pix\n10-shot', 'generated', ('pix2pix', 10), PIX_COLOR),
    ('Pix2Pix\n50-shot', 'generated', ('pix2pix', 50), PIX_COLOR),
    ('CycleGAN\n5-shot', 'generated', ('cyclegan', 5), CYCLE_COLOR),
    ('CycleGAN\n10-shot', 'generated', ('cyclegan', 10), CYCLE_COLOR),
    ('CycleGAN\n50-shot', 'generated', ('cyclegan', 50), CYCLE_COLOR),
]


def image_path(filename, kind, model_shot):
    if kind == 'input':
        return TEST_ROOT / 'test_A' / filename
    if kind == 'target':
        return TEST_ROOT / 'test_B' / filename
    model, shot = model_shot
    return GENERATED_ROOT / model / f'{shot}shot' / 'seed1' / filename


def plot_grid(roles, output_name, *, show=True):
    roles = list(roles)
    figure_height = 1.95 * len(roles) + 1.45
    fig, axes = plt.subplots(
        len(roles), len(CONDITIONS),
        figsize=(19.2, figure_height),
        squeeze=False,
    )

    for row_index, role in enumerate(roles):
        filename = selection.loc[role, 'filename']
        for column_index, (title, kind, model_shot, color) in enumerate(CONDITIONS):
            ax = axes[row_index, column_index]
            path = image_path(filename, kind, model_shot)
            if not path.is_file():
                raise FileNotFoundError(path)
            with Image.open(path) as image:
                ax.imshow(image.convert('RGB'))
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_color(color)
                spine.set_linewidth(2.3)
            if row_index == 0:
                ax.set_title(title, color=color, fontsize=11.5, fontweight='bold', pad=7)
            if column_index == 0:
                ax.set_ylabel(
                    ROLE_LABELS[role], fontsize=12.5, fontweight='bold',
                    rotation=90, labelpad=11, color='#111827',
                )

    fig.suptitle(
        'Qualitative comparison on fixed held-out scenes',
        fontsize=16, fontweight='bold', y=0.985,
    )
    fig.text(
        0.5, 0.012,
        'Training seed 1 · inference seed 0 · identical filenames across conditions',
        ha='center', va='bottom', fontsize=10.5, color='#4B5563',
    )
    fig.subplots_adjust(left=0.055, right=0.995, top=0.88, bottom=0.075, wspace=0.025, hspace=0.11)
    output_path = FIGURE_DIR / output_name
    fig.savefig(output_path, dpi=240, bbox_inches='tight', facecolor='white')
    print(f'Wrote {output_path}')
    if show:
        plt.show()
    plt.close(fig)
    return output_path

## Slide-ready grids

The first export contains all three selection roles. The second is the recommended compact slide version: it retains a representative case and the case directly tied to the breaking-point analysis.

In [ ]:
all_cases_path = plot_grid(
    ROLE_ORDER,
    'qualitative_all_cases.png',
)

slide_path = plot_grid(
    ['typical', 'breaking_sensitive'],
    'qualitative_slide_compact.png',
)

## Individual case exports

These files are useful when a report or slide layout needs only one row. They are exported without adding three more large notebook outputs.

In [ ]:
individual_paths = {
    role: plot_grid([role], f'qualitative_{role}.png', show=False)
    for role in ROLE_ORDER
}
individual_paths